# DQN with ALE/MsPacman-v5

This notebook loads [config.yaml](config.yaml) and trains the `QAgent` on `ALE/MsPacman-v5` using the Gymnasium ALE API.

Ms. Pac-Man is a pixel-based Atari env: the observation is a `(210, 160, 3)` RGB frame and the action space is `Discrete(9)`. We apply the same Mnih et al. (2015) preprocessing pipeline as `dqn_enduro` — grayscale, 84×84, 4-frame skip, 4-frame stack — giving a `(4, 84, 84)` uint8 input to the same Nature DQN CNN.

The one meaningful difference from Enduro is `terminal_on_life_loss=True`: Ms. Pac-Man has 3 lives and treating each ghost-death as a terminal signal is standard practice (Mnih et al. 2015 Appendix) — it gives the agent a clear negative signal for dying rather than letting it coast through lost lives mid-episode.

## Imports

In [ ]:
import sys, pathlib, yaml
from functools import partial
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

import ale_py
import gymnasium as gym
gym.register_envs(ale_py)

SRC = pathlib.Path.cwd().parents[1] / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from environments.base_env import make_environment
from agents.q_agent import QAgent
from training import dqn_training_loop
from analysis.low_rank.hankel_policy import hankel_rollout
from analysis.low_rank.rank import plot_matrix_spectra, row_rank_property_check
from analysis.visualisations.heatmaps import plot_matrix_heatmap

## Reading the config file

In [ ]:
with open("config.yaml") as f:
    cfg = yaml.safe_load(f)

dev = cfg["experiment"]["device"]
if dev == "cuda":
    device = "cuda" if torch.cuda.is_available() else "cpu"
elif dev == "mps":
    device = "mps" if torch.backends.mps.is_available() else "cpu"
else:
    device = dev

seed = cfg["experiment"]["seed"]
torch.manual_seed(seed)
np.random.seed(seed)

print("device:", device)
cfg

## Creating the Environment

The `atari` block triggers the Atari branch of `make_environment`, wiring `AtariPreprocessing` followed by `FrameStackObservation`. Resulting `observation_space` is `Box(0, 255, (4, 84, 84), uint8)`.

In [ ]:
env_cfg = cfg["environment"]
env = make_environment(
    env_cfg["name"],
    render_mode=env_cfg["render_mode"],
    discrete_config=env_cfg["discrete_config"],
    normalise=env_cfg["normalise"],
    clip=env_cfg["clip"],
    atari=env_cfg["atari"],
)
obs_shape = env.observation_space.shape   # (4, 84, 84)
n_actions = env.action_space.n
print("obs_shape:", obs_shape, "dtype:", env.observation_space.dtype, "n_actions:", n_actions)

## CNN Q-network

Same Nature DQN architecture as `dqn_enduro` — no changes needed. The network is game-agnostic: `n_actions` is read from the env at runtime.

| Layer | Spec | Output |
|---|---|---|
| Conv1 | 4 → 32, kernel 8, stride 4 | 32×20×20 |
| Conv2 | 32 → 64, kernel 4, stride 2 | 64×9×9 |
| Conv3 | 64 → 64, kernel 3, stride 1 | 64×7×7 |
| FC1   | 3136 → 512 | 512 |
| FC2   | 512 → n_actions | Q-values |

In [ ]:
class NatureCNN(nn.Module):
    """Maps a (C, 84, 84) frame stack to Q-values of shape (n_actions,)."""
    def __init__(self, in_channels, n_actions, fc_hidden=512):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=8, stride=4), nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2),          nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1),          nn.ReLU(),
            nn.Flatten(),
        )
        with torch.no_grad():
            dummy = torch.zeros(1, in_channels, 84, 84)
            flat_dim = self.features(dummy).shape[1]
        self.head = nn.Sequential(
            nn.Linear(flat_dim, fc_hidden), nn.ReLU(),
            nn.Linear(fc_hidden, n_actions),
        )

    def forward(self, x):
        x = x.float() / 255.0
        return self.head(self.features(x))

## Creating the Agent

In [ ]:
a = cfg["agent"]
nn_extra_kwargs = {
    "in_channels": obs_shape[0],
    "n_actions": n_actions,
    "fc_hidden": cfg["network"]["fc_hidden"],
}
agent = QAgent(
    replay_buffer_capacity=a["replay_buffer_capacity"],
    q_network=NatureCNN,
    batch_size=a["batch_size"],
    nn_learning_rate=a["nn_learning_rate"],
    nn_extra_kwargs=nn_extra_kwargs,
    env=env,
    eps_start=a["eps_start"],
    eps_min=a["eps_min"],
    decay_rate=a["decay_rate"],
    discount_factor=a["discount_factor"],
    device=device,
    TD_LR=a["TD_LR"],
    buffer_util=a["buffer_util"],
    gd_steps_ceil=a["gd_steps_ceil"],
    grad_clip_norm=a["grad_clip_norm"],
    double=a["double"]
)

## Analysis (Low Rank)

Only `hankel_rollout` is wired here — `q_matrix_dqn` discretises each observation dimension into bins, which is meaningless for `(4, 84, 84)` pixel stacks.

In [ ]:
analysis = cfg["analysis"]

analysis_methods = {
    "hankel_rollout": hankel_rollout,
}

analysis["methods"] = [
    (partial(analysis_methods[m["name"]], **m.get("kwargs", {})), m["outputs"])
    for m in analysis["methods"]
]
analysis

## Agent Training

Ms. Pac-Man episodes are shorter than Enduro on average (3 lives, quicker deaths early on), but the maze structure means the agent needs many episodes to explore effectively. Expect reward to climb slowly past the random baseline (~200) before showing consistent improvement.

In [ ]:
t = cfg["training"]
rewards = dqn_training_loop(
    agent, env,
    no_episodes=t["no_episodes"],
    target_network_update_steps=t["target_network_update_steps"],
    train_frequency_steps=t["train_frequency_steps"],
    use_episode_training=t["use_episode_training"],
    solved_reward=t["solved_reward"],
    warmup_steps=t["warmup_steps"],
    early_stopping_patience_eps=t["early_stopping_patience_eps"],
    no_eps_to_avg=t["no_eps_to_avg"],
    np_seed=seed,
    analysis_config=analysis,
    DEBUG=False, atari=True
)

## Training and Analysis Plots

In [ ]:
rewards = np.asarray(rewards, dtype=float)
plt.figure(figsize=(8, 4))
plt.plot(rewards, alpha=0.35, label="episode reward")
if len(rewards) >= 10:
    k = 10
    ma = np.convolve(rewards, np.ones(k) / k, mode="valid")
    plt.plot(range(k - 1, len(rewards)), ma, label=f"{k}-ep moving avg")
plt.xlabel("episode"); plt.ylabel("total reward"); plt.title("DQN on ALE/MsPacman-v5")
plt.legend(); plt.show()

In [ ]:
for method, names in analysis["methods"]:
    results = method(agent=agent, env=env)
    if not isinstance(results, tuple):
        results = (results,)
    for matrix, name in zip(results, names):
        print(name)
        plot_matrix_heatmap(matrix, name)
        r, sr, spk, shape, irs, ics, rc, cc, nzr, nzc = row_rank_property_check(matrix, name)
        print(f"eff_rank: {r}, stable_rank: {sr:.2f}, spikiness: {spk:.2f}, shape: {shape}, non-zero rows :{nzr}, non-zero cols:{nzc}")
        print(f"top-r leverage spread: row min={irs.min():.4g} max={irs.max():.4g} (uniform {1.0/shape[0]:.4g}) | col min={ics.min():.4g} max={ics.max():.4g} (uniform {1.0/shape[1]:.4g})")
        print(f"coherence score: row={rc:.4g} col={cc:.4g}")

## Greedy rollout video

Record one greedy (`act_greedy`) episode of the trained agent and display it inline.

In [ ]:
from analysis.visualisations.rollout_video import record_greedy_episode
from IPython.display import Video
import glob

video_dir = "videos"
eval_env = make_environment(
    env_cfg["name"],
    render_mode="rgb_array",
    discrete_config=env_cfg["discrete_config"],
    normalise=env_cfg["normalise"],
    clip=env_cfg["clip"],
    atari=env_cfg["atari"],
)
prefix = record_greedy_episode(agent, eval_env, video_dir, episode=0, seed=seed)
mp4 = sorted(glob.glob(f"{video_dir}/{prefix}-*.mp4"))[-1]
print("saved:", mp4)
Video(mp4, embed=True)